We want to know if a cell has consistent tuning across conditions or between homing/escpae and explore

1. compute tuning
2. compute linear sift tuning 
3. if curve is significant in both conditions
4. bootstrap for similarity metric (subsample many times from both the homing and exploration period and compute the firing by distance. Then I take pairs of subsampled tuning curves and compute some metric of similarity (e.g. difference in the preferred distance or cosine similarity) and that gives me a whole distribution of the metrics. If 0 falls outside the 95% of that distribution then the two curves are different.) 

In [ ]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

#JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
# 
experiments_objects = [JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, # JAL6_flip7_1apr, # this session is sus
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

In [ ]:
%load_ext autoreload
from JR_test_scripts.escape.escape_utils import load, load_homing, firing_by_bin, smooth_firing_by_bin_by_trial
from behave_analysis.utils.creating_directories import make_directory
from JR_test_scripts.escape.escape_data_loading_funcs import extract_homing_and_escape_periods, build_shift_vector, compute_dist_shelt
from JR_test_scripts.escape.escape_tuning_funcs import tuning_method, leave_one_out_reliability
from behave_analysis.utils.creating_directories import make_directory
from scipy.ndimage import gaussian_filter1d
from JR_test_scripts.escape.escape_plotting_funcs import new_plot_linear_shift, plot_reliability
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
"""Linear shift significance for tuning curves
THIS IS THE GOOD ONE!!"""
%autoreload 2
"""Compute real statistics (on the full session)"""
# compression = ['bird_dist_shelter','bird_dist_first_goal', 'full_distance_shelter', 'escape', 'escape_shelter', 'escape_first_goal']
compression = ['bird_dist_shelter','escape']
Nbins = 25

for exp in experiments_objects:

    print(exp.nick_name + '_' + exp.experiment_date)

    session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
    ons, offs, homie = load_homing(session, len(behave))
    fcm = gaussian_filter1d(frame_by_cluster_matrix, 2, axis = 0)

    for comp in compression:

        # what are the bins supposed to be for 'bird_dist_shelter'?
        if comp == 'bird_dist_shelter':
            # var = compute_dist_shelt(x_pos, y_pos, cond=np.zeros_like(x_pos), session=session)
            # bins = np.arange(0,np.amax(var),np.amax(var)/Nbins)
            bins = np.append(np.arange(0,925,925/Nbins), 925)
        elif comp == 'escape':
            bins = np.append(np.arange(0,1,1/Nbins), 1+1e-10)
        
        nickname = exp.nick_name + '_' + exp.experiment_date
        exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp

        # the real stat
        var, escape_matrix, cond, h_start = extract_homing_and_escape_periods(session, 
                                                                                fcm, 
                                                                                behave, 
                                                                                y_pos, x_pos, 
                                                                                bar, 
                                                                                barflip, 
                                                                                comp, 
                                                                                ons, offs, 
                                                                                bins=bins,
                                                                                no_stationary = False, 
                                                                                return_escape = False,
                                                                                zscore = False)

        # initialize vars
        n_neur = fcm.shape[1]
        n_cond = len(np.unique(cond))

        _, _, full_reliability = leave_one_out_reliability(var, escape_matrix, cond, h_start, Nbins, n_cond, n_neur)

        y_fitted_full, R_full, fr_full, params_full, mat_full_cond = tuning_method(var, 
                                                                                    escape_matrix, 
                                                                                    cond, 
                                                                                    h_start, 
                                                                                    Nbins, 
                                                                                    n_cond,
                                                                                    n_neur,
                                                                                    fitting = False)

        """Compute real statistics (on the central third of each condition)"""

        # setting up the shifts
        min_step = 120
        step = 400
        step_n = 100
        shifts_one_sided = np.arange(min_step,min_step+((step_n/2)*step), step)

        shifts, shift_vector = build_shift_vector(bar, barflip, session, ons, offs, shifts_one_sided)

        # the real stat
        var, escape_matrix, cond, h_start = extract_homing_and_escape_periods(session, 
                                                                                fcm[shift_vector,:], 
                                                                                behave[shift_vector], 
                                                                                y_pos[shift_vector], 
                                                                                x_pos[shift_vector], 
                                                                                bar[shift_vector], # should this also be shifted?!?! 
                                                                                barflip[shift_vector], 
                                                                                comp, 
                                                                                ons, offs, shift_vector, 
                                                                                bins=bins,
                                                                                no_stationary = False, 
                                                                                return_escape = False,
                                                                                zscore = False)

        y_fitted_real, R_real, fr_real, params_real, mat_real_cond = tuning_method(var, 
                                                                                escape_matrix, 
                                                                                cond, 
                                                                                h_start, 
                                                                                Nbins, 
                                                                                n_cond,
                                                                                n_neur,
                                                                                fitting = False)

        """Compute shifted statistics"""

        # initialize variables for output
        y_fitted_shift = np.full((step_n, n_cond, n_neur, Nbins), np.nan) # conditions x neurons x n_bins
        R_shift = np.zeros((step_n,n_neur, n_cond)) # neurons x conditions
        params_shifts = np.zeros((step_n,n_neur, n_cond, 6)) # neurons x conditions
        fr_shift = np.full((step_n, n_cond, n_neur, Nbins), np.nan)
        c = [len([x for x in h_start if cond[x] == i]) for i in range(3)] # trial n per condition
        mat_shift_cond = np.full((step_n, n_cond, n_neur, max(c), Nbins), np.nan)

        for s_idx, s in enumerate(shifts):
            shifted_vec = np.roll(shift_vector,int(s))
            var, escape_matrix, cond, h_start = extract_homing_and_escape_periods(session, 
                                                                                    fcm[shifted_vec,:], 
                                                                                    behave[shift_vector], 
                                                                                    y_pos[shift_vector], 
                                                                                    x_pos[shift_vector], 
                                                                                    bar[shift_vector], # should this also be shifted?!?! 
                                                                                    barflip[shift_vector], 
                                                                                    comp, 
                                                                                    ons, offs, shift_vector, 
                                                                                    bins=bins,
                                                                                    no_stationary = False, 
                                                                                    return_escape = False, 
                                                                                    zscore = False)

            y_fitted_shift[s_idx,:,:,:], R_shift[s_idx,:,:], fr_shift[s_idx,:,:,:], params_shifts[s_idx,:,:,:], mat_shift_cond[s_idx,:,:,:] = tuning_method(var, 
                                                                                                                                        escape_matrix, 
                                                                                                                                        cond, 
                                                                                                                                        h_start, 
                                                                                                                                        Nbins, 
                                                                                                                                        n_cond,
                                                                                                                                        n_neur,
                                                                                                                                        fitting = False)

        """Save data"""
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/")
        saving_file = dump_path + exp_nickname + '_ProperTuning_'+ str(Nbins) + 'bins'
        np.savez(saving_file, 
                R_shift=R_shift, params_shifts=params_shifts, y_fitted_shift=y_fitted_shift, 
                fr_shift=fr_shift, mat_shift_cond=mat_shift_cond,
                R_full=R_full, params_full=params_full, y_fitted_full=y_fitted_full, 
                fr_full=fr_full, mat_full_cond=mat_full_cond, full_reliability=full_reliability,
                R_real=R_real, params_real=params_real, y_fitted_real=y_fitted_real, 
                fr_real=fr_real, mat_real_cond=mat_real_cond, bins=bins)


        """Plot linear shift and real stats"""
        colors = ['#228B22','#FF8C00','#008B8B']
        c_names = ['shelter_only', 'barrier', 'barrier_flipped']
        nickname = exp.nick_name + '_' + exp.experiment_date
        exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp + '_'+ str(Nbins) + 'bins'
        dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + "/" + exp_nickname)
        
        new_plot_linear_shift(y_fitted_shift, y_fitted_real, y_fitted_full, params_shifts, params_real, fr_full, fr_shift, params_full, comp, n_neur, n_cond, colors, c_names, dump_path, name = '_new')
        
        # """Plotting reliability"""

        plot_reliability(mat_full_cond, fr_full, full_reliability, comp, colors, c_names, n_cond, n_neur, dump_path)

In [ ]:
"""Plotting reliability"""
import time

def save_figure(fig, path):
    fig.savefig(path, dpi=100)
    plt.close(fig)  # Close to free memory

start = time.time()
nickname = exp.nick_name + '_' + exp.experiment_date
exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + "/" + exp_nickname)

n_neur = fr_full.shape[1]
n_cond = fr_full.shape[0]

# compute min/max for this neuron across all conditions
vmin, vmax = np.nanmin(mat_full_cond, axis=(0, 2, 3)), np.nanmax(mat_full_cond, axis=(0, 2, 3))  # Ignore NaNs

# compute min/max for the average also
ymin, ymax = [np.nanmin(fr_full, axis=(0, 2)), np.nanmax(fr_full, axis=(0, 2))]

set_up = time.time()
print('Set up time: ' + str(set_up - start))

for neur in range(3):
    start = time.time()
    fig, axs = plt.subplots(1,3, figsize = (12,4), constrained_layout=True)

    ylim = [ymin[neur], ymax[neur]]
    if ylim == [0,0]: ylim = [0,1]

    for c in range(n_cond):
        nan_rows = np.all(np.isnan(mat_full_cond[c,neur,:,:]), axis=1)
        im = axs[c].imshow(mat_full_cond[c,neur,~nan_rows,:], cmap="gray_r", vmin = vmin[neur], vmax = vmax[neur], aspect="auto", interpolation = "none")
        axs[c].set_title(c_names[c] + f'\n Reliability = {full_reliability[c,neur]:.2f}')
        axs[c]. set_xlabel(comp)
        if c == 0:
            axs[c].set_ylabel('trials')

        ax2 = axs[c].twinx()
        ax2.plot(fr_full[c,neur,:], linewidth = 2, color = colors[c])
        ax2.spines["right"].set_color(colors[c])
        ax2.tick_params(axis="y", colors=colors[c])  # Change tick color
        ax2.yaxis.label.set_color(colors[c])  # Change axis label color
        ax2.set_ylim(ylim)
        
        if c == 2:
            ax2.set_ylabel('Firing rate')
            cbar = fig.colorbar(im, ax=axs[c], location="right", pad=0.1)
            cbar.set_label('Firing rate')

        fig.savefig(dump_path + "/neuron" + str(neur) + "_loo_reliability.png")
        plt.close()
    eachneur = time.time()
    print('This neuron took: '+ str(eachneur - start))

In [ ]:
"""Plot linear shift and real stats"""
%matplotlib inline
condition = [0,1,2]
colors = ['#228B22','#FF8C00','#008B8B']
c_names = ['shelter_only', 'barrier', 'barrier_flipped']
nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)

for neuron in np.arange(np.shape(frame_by_cluster_matrix)[1]):
    fig, axs = plt.subplots(3,3,figsize = (12,12))
    # compute min/max for the average also
    ylim = [0,0]
    ylim[0] = np.nanmin((np.nanmin(y_fitted_shift[:,:,neuron,:]), np.nanmin(y_fitted_real[:,neuron,:])))
    ylim[1] = np.nanmax((np.nanmax(y_fitted_shift[:,:,neuron,:]), np.nanmax(y_fitted_real[:,neuron,:])))

    for c in condition:
        # fit in real vs shifted trials
        axs[c,0].plot(y_fitted_shift[:,c,neuron,:].T,'k', alpha = .3)
        axs[c,0].plot(y_fitted_real[c,neuron,:],color = colors[c], linewidth = 3)
        # axs[c,0].plot(fr_shift[:,c,neuron,:].T,'k', alpha = .3)
        # axs[c,0].plot(fr_real[c,neuron,:],color = colors[c], linewidth = 3)
        axs[c,0].set_title(c_names[c])
        axs[c,0].set_ylabel('Firing rate')
        axs[c,0].set_ylim(ylim)
        axs[c,0].set_xlabel(comp)

        # fit amplitude
        axs[c,1].hist(params_shifts[:,neuron,c,0], edgecolor = None, facecolor = 'k', alpha = .3)
        yl = axs[c,1].get_ylim()
        axs[c,1].plot([params_real[neuron, c, 0],params_real[neuron, c, 0]],yl,color = colors[c])
        axs[c,1].set_xlabel('Firing rate')
        axs[c,0].set_xlim(ylim)
        axs[c,1].set_ylabel('Linear shifts')
        if params_real[neuron, c, 0] > np.percentile(params_shifts[:,neuron,c,0],95):
            axs[c,1].set_title('Fit amp. 95th perc: sig')
        else:
            axs[c,1].set_title('Fit amp. 95th perc: not sig')
        
        # goodness of fit
        axs[c,2].hist(R_shift[:,neuron,c], edgecolor = None, facecolor = 'k', alpha = .3)
        yl = axs[c,2].get_ylim()
        axs[c,2].plot([R_real[neuron, c],R_real[neuron, c]],yl,color = colors[c])
        if R_real[neuron,c] > np.percentile(R_shift[:,neuron,c],95):
            axs[c,2].set_title('Fit goodness 95th perc.: sig')
        else:
            axs[c,2].set_title('Fit goodness 95th perc.: not sig')
        axs[c,2].set_xlabel('R^2')
        axs[c,2].set_ylabel('Linear shifts')

    plt.tight_layout()
    fig.savefig(dump_path + "/neuron" + str(neuron) + "_linshit.png")
    plt.close()

How to compare cells with significant tuning curves!?
- for a given condition, pull out the significant ones: do they tile the space?
- sig cells are found using the fits for the central third, plotting is done with the full fits
- pull out neurons that are significant in more than one condition: compare their tuning! mu? cosine similarity?

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
condition = [0,1,2]
c_names = ['shelter_only', 'barrier', 'barrier_flipped']
fig, axs = plt.subplots(4,4,figsize = (12,12))

# top row: the significant neurons in each condition (from linshit)
for c in condition:
    sig_this_c = R_real[:,c] > np.percentile(R_shift[:,:,c],95, axis = 0)
    sig_fit = y_fitted_full[c,sig_this_c,:]
    sig_mu = params_full[sig_this_c,c,1]
    axs[0,c].imshow(sig_fit[np.argsort(sig_mu),:], cmap="gray_r", vmin = 0, vmax = 1.5, aspect="auto", interpolation = "none")
    axs[0,c].set_title(c_names[c])
    axs[0,c].set_ylabel('Neurons with sig. tuning')
    axs[0,c].set_xlabel(comp)
fig.delaxes(axs[0, 3])

# rows 1-3: tuning of neurons significant in more than one condition
comp_c = [[0,0,1],[1,2,2]]
for c in condition:
    sig_intersect = np.logical_and(R_real[:,comp_c[0][c]] > np.percentile(R_shift[:,:,comp_c[0][c]],95, axis = 0),
                    R_real[:,comp_c[1][c]] > np.percentile(R_shift[:,:,comp_c[1][c]],95, axis = 0))
    sig_fit1 = y_fitted_full[comp_c[0][c],sig_intersect,:]
    sig_fit2 = y_fitted_full[comp_c[1][c],sig_intersect,:]
    sig_mu1 = params_full[sig_intersect,comp_c[0][c],1]
    sig_mu2 = params_full[sig_intersect,comp_c[1][c],1]
    # show the tuning curves
    axs[c+1,0].imshow(sig_fit1[np.argsort(sig_mu1),:], cmap="gray_r", vmin = 0, vmax = 1.5, aspect="auto", interpolation = "none")
    axs[c+1,0].set_title(c_names[comp_c[0][c]])
    axs[c+1,0].set_ylabel('Neurons with sig. tuning\n' + c_names[comp_c[0][c]] + ' and ' + c_names[comp_c[0][c]])
    axs[c+1,0].set_xlabel(comp)
    axs[c+1,1].imshow(sig_fit2[np.argsort(sig_mu1),:], cmap="gray_r", vmin = 0, vmax = 1.5, aspect="auto", interpolation = "none")
    axs[c+1,1].set_title(c_names[comp_c[1][c]])
    axs[c+1,1].set_xlabel(comp)
    # compare mu
    axs[c+1,2].plot([0,sig_fit1.shape[1]],[0,sig_fit1.shape[1]],'--k')
    axs[c+1,2].scatter(sig_mu1,sig_mu2)
    axs[c+1,2].set_xlabel('Fitted gaussian mu\n ' + c_names[comp_c[0][c]])
    axs[c+1,2].set_ylabel('Fitted gaussian mu\n ' + c_names[comp_c[1][c]])
    # cosine similarity
    cos_sim = cosine_similarity(sig_fit1, sig_fit2, dense_output=True) 
    cos_sim_diag = np.diag(cos_sim)
    axs[c+1,3].hist(cos_sim_diag)
    axs[c+1,3].set_xlabel('Cosine Similairity')

plt.tight_layout()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(sig_fit1, sig_fit2, dense_output=True) 
cos_sim_diag = np.diag(cos_sim)

idx = 2
print(cos_sim_diag[idx], cosine_similarity(sig_fit1[idx,:].reshape(1, -1),sig_fit2[idx,:].reshape(1, -1))[0,0])

In [ ]:
"""Linear shift significance for tuning curves
This method shifts after creating the escape_matrix - NOT the ideal approach
DEPRECATED"""
neuron = 17
condition = 0

bins = int(np.amax(var)+1)
X = escape_matrix[neuron,cond == condition]
this_cond = cond[cond == condition]
cond_start = [x for x in h_start if cond[x] == condition]
central_y = var[cond == condition]
n_central_homies = np.floor(len(cond_start)/3).astype(int)
central_chunk = [cond_start[n_central_homies+1], cond_start[(n_central_homies*2)+1]] # the indices for the beginning and end of the chunk we're actually using
central_start = cond_start[n_central_homies+1:(n_central_homies*2)+2] # the indices for the start of homings in the chunk we're using
central_start = [x - central_start[0] for x in central_start]

# set up shifts
min_step = 40
step_size = min_step*3
# ensure 0 step is not included (thanks to min step away from the 0 stat!)
shifts_left = - np.arange(min_step, central_chunk[0], step_size)
shifts_right = np.arange(min_step, len(X) - central_chunk[1], step_size)
# now double them so we go in both directions
shifts = np.sort(np.hstack((shifts_left,shifts_right)))

all_fr = np.full((len(shifts),bins), np.nan)
all_R = np.full(len(shifts), np.nan)

for idx, s in enumerate(shifts):

    # take only the central chunk!
    y = central_y[int(s + central_chunk[0]) : int(s + central_chunk[1])]
    n = X[central_chunk[0]:central_chunk[1]]

    # extract firing per bin for each shift
    matrix = np.full((len(central_start),bins), np.nan)
    # iterate through trials, pull out firing by bin
    for tr, _ in enumerate(central_start[:-1]):
        neur = X[central_start[tr]:central_start[tr+1]]
        v = y[central_start[tr]:central_start[tr+1]]
        matrix[tr,:] = firing_by_bin(v.astype(int), neur, bins, remove_empty = False)

    # smooth activity on each trial
    # remove trials that are all nan
    all_nan_rows = np.all(np.isnan(matrix), axis=1)
    all_test = matrix[~all_nan_rows,:]
    smooth_test = np.full_like(matrix, np.nan)
    for sidx, sxm in enumerate(matrix):
        this_line = np.full_like(sxm, np.nan)
        where_nan = np.isnan(sxm)
        this_line[~where_nan] = gaussian_filter1d(sxm[~where_nan], 3)
        smooth_test[sidx,:] = this_line

    # get median firing across trials
    all_nan_cols = np.all(np.isnan(smooth_test), axis=0)
    smoothed_firing_rates = np.full(len(all_nan_cols), np.nan)
    smoothed_firing_rates[~all_nan_cols] = np.nanmedian(smooth_test[:,~all_nan_cols], axis = 0)

    # make firing rates positive
    shift_constant = abs(np.nanmin(smoothed_firing_rates))+ 1e-6 # Add a small epsilon to avoid exact zero
    smoothed_firing_rates = smoothed_firing_rates + shift_constant
    distances = np.arange(len(smoothed_firing_rates))

    R, y, params, double_wins = gaussian_fitting(smoothed_firing_rates[~np.isnan(smoothed_firing_rates)], np.arange(np.sum(~np.isnan(smoothed_firing_rates))), verbose = False)
    all_R[idx] = R
    all_fr[idx,:] = smoothed_firing_rates

In [ ]:
"""Linear shift significance for tuning curves
OLDER DEPRECATE VERSION!!"""

"""Compute real statistics (on the full session)"""

fcm = gaussian_filter1d(frame_by_cluster_matrix, 2, axis = 0)

# the real stat
var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, 
                                                                                 fcm, 
                                                                                 behave, 
                                                                                 y_pos, x_pos, 
                                                                                 bar, 
                                                                                 barflip, 
                                                                                 comp, 
                                                                                 ons, offs, 
                                                                                 no_stationary = False, 
                                                                                 return_escape = True,
                                                                                 zscore = False)

# initialize vars
bins = int(np.amax(var)+1)
n_neur = frame_by_cluster_matrix.shape[1]
n_cond = len(np.unique(cond))

# compute xval tuning during homing/escape
mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start, bins = bins, averaging = 'mean')

# initialize variables for output
y_fitted_full = np.full((n_cond, n_neur, bins), np.nan) # conditions x neurons x n_bins
R_full = np.zeros((n_neur, n_cond)) # neurons x conditions
fr_full = np.zeros((n_cond, n_neur, bins)) # conditions x neurons x n_bins
params_full = np.full((n_neur, n_cond, 6), np.nan)

for neuron in np.arange(n_neur):
    for condition in np.arange(n_cond):

        # process the firing by bin for each trial for this neuron in this 
        matrix = mat_by_cond[condition][neuron,:,:]
        distances, smoothed_firing_rates, shift_constant, smooth_test = smooth_firing_by_bin_by_trial(matrix, fr_all_time = [], method = 'across_trials')
        R, y, params, double_wins = gaussian_fitting(smoothed_firing_rates[~np.isnan(smoothed_firing_rates)], np.arange(np.sum(~np.isnan(smoothed_firing_rates))), verbose = False)
        y_fitted = np.full_like(smoothed_firing_rates, np.nan)
        y_fitted[~np.isnan(smoothed_firing_rates)] = y

        # dump parameters into the variables to return
        R_full[neuron, condition] = R
        params_full[neuron, condition,:len(params)] = params
        y_fitted_full[condition, neuron, :len(y_fitted)] = y_fitted
        fr_full[condition, neuron, :len(smoothed_firing_rates)] = smoothed_firing_rates

"""Compute real statistics (on the central third of each condition)"""

# setting up the shifts
T = np.shape(frame_by_cluster_matrix)[0]
central_chunk = T/3
N = int((T - central_chunk)/2)
min_step = 120
step = 400
step_n = 100
shifts_one_sided = np.arange(min_step,min_step+((step_n/2)*step), step)

shelter = np.where(bar == True)[0][0]
bar_in = np.where(barflip == True)[0][0]
mid_shelter = [int(shelter/3), int((shelter/3)*2)]
mid_bar = [int(shelter+((bar_in - shelter)/3)), int(shelter+(((bar_in - shelter)/3)*2))]
mid_flip = [int(bar_in+((len(bar) - bar_in)/3)), int(bar_in+(((len(bar) - bar_in)/3)*2))]
shift_vector = np.zeros(len(bar))
shift_vector[mid_shelter[0]:mid_shelter[1]] = 1
shift_vector[mid_bar[0]:mid_bar[1]] = 1
shift_vector[mid_flip[0]:mid_flip[1]] = 1
shift_vector = shift_vector.astype(bool)

# make sure we're not shifting our of range
shifts_left = shifts_one_sided[shifts_one_sided < mid_shelter[0]]
shifts_right = shifts_one_sided[(shifts_one_sided + mid_flip[1]) < len(bar)]
# now double them so we go in both directions
shifts = np.sort(np.hstack((shifts_right,-shifts_left)))

# the real stat
var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, 
                                                                                fcm[shift_vector,:], 
                                                                                behave[shift_vector], 
                                                                                y_pos[shift_vector], 
                                                                                x_pos[shift_vector], 
                                                                                bar[shift_vector], # should this also be shifted?!?! 
                                                                                barflip[shift_vector], 
                                                                                comp, 
                                                                                ons, offs, shift_vector, 
                                                                                no_stationary = False, 
                                                                                return_escape = True,
                                                                                zscore = False)
    
# compute xval tuning during homing/escape
mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start, bins = bins, averaging = 'mean')

# initialize variables for output
y_fitted_real = np.full((n_cond, n_neur, bins), np.nan) # conditions x neurons x n_bins
R_real = np.zeros((n_neur, n_cond)) # neurons x conditions
fr_real = np.zeros((n_cond, n_neur, bins)) # conditions x neurons x n_bins
params_real = np.full((n_neur, n_cond, 6)np.nan)

for neuron in np.arange(n_neur):
    for condition in np.arange(n_cond):

        # process the firing by bin for each trial for this neuron in this 
        matrix = mat_by_cond[condition][neuron,:,:]
        distances, smoothed_firing_rates, shift_constant, smooth_test = smooth_firing_by_bin_by_trial(matrix, fr_all_time = [], method = 'across_trials')
        R, y, params, double_wins = gaussian_fitting(smoothed_firing_rates[~np.isnan(smoothed_firing_rates)], np.arange(np.sum(~np.isnan(smoothed_firing_rates))), verbose = False)
        y_fitted = np.full_like(smoothed_firing_rates, np.nan)
        y_fitted[~np.isnan(smoothed_firing_rates)] = y

        # dump parameters into the variables to return
        R_real[neuron, condition] = R
        params_real[neuron, condition,:len(params)] = params
        y_fitted_real[condition, neuron, :len(y_fitted)] = y_fitted
        fr_real[condition, neuron, :len(smoothed_firing_rates)] = smoothed_firing_rates

"""Compute shifted statistics"""

# initialize variables for output
y_fitted_shift = np.full((step_n, n_cond, n_neur, bins), np.nan) # conditions x neurons x n_bins
R_shift = np.zeros((step_n,n_neur, n_cond)) # neurons x conditions
params_shifts = np.zeros((step_n,n_neur, n_cond, 6)) # neurons x conditions
fr_shift = np.full((step_n, n_cond, n_neur, bins), np.nan)

for s_idx, s in enumerate(shifts):
    shifted_vec = np.roll(shift_vector,int(s))
    var, escape_matrix, cond, esc_start, h_start = extract_homing_and_escape_periods(session, 
                                                                                    fcm[shift_vector,:], 
                                                                                    behave[shifted_vec], 
                                                                                    y_pos[shifted_vec], 
                                                                                    x_pos[shifted_vec], 
                                                                                    bar[shifted_vec], # should this also be shifted?!?! 
                                                                                    barflip[shifted_vec], 
                                                                                    comp, 
                                                                                    ons, offs, shifted_vec, 
                                                                                    no_stationary = False, 
                                                                                    return_escape = True, 
                                                                                    zscore = False)

    mat_by_cond = single_trial_tuning(escape_matrix, var, cond, h_start, bins = bins, averaging = 'mean')
    for neuron in np.arange(n_neur):
        for condition in np.arange(n_cond):

            # process the firing by bin for each trial for this neuron in this 
            matrix = mat_by_cond[condition][neuron,:,:]
            distances, smoothed_firing_rates, shift_constant, smooth_test = smooth_firing_by_bin_by_trial(matrix, fr_all_time = [], method = 'across_trials')
            R, y, params, double_wins = gaussian_fitting(smoothed_firing_rates[~np.isnan(smoothed_firing_rates)], np.arange(np.sum(~np.isnan(smoothed_firing_rates))), verbose = False)
            y_fitted = np.full_like(smoothed_firing_rates, np.nan)
            y_fitted[~np.isnan(smoothed_firing_rates)] = y

            # dump parameters into the variables to return
            R_shift[s_idx, neuron, condition] = R
            params_shifts[s_idx, neuron, condition,:len(params)] = params
            y_fitted_shift[s_idx, condition, neuron, :len(y_fitted)] = y_fitted
            fr_shift[s_idx, condition, neuron, :len(smoothed_firing_rates)] = smoothed_firing_rates

nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/"+nickname)
saving_file = dump_path + nickname + '_TrialMean'
np.savez(saving_file, R_shift, params_shifts, y_fitted_shift, fr_shift,
                        R_full, params_full, y_fitted_full, fr_full,
                        R_real, params_real, y_fitted_real, fr_real)